In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Specificarea opțiunilor Sampler

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';







<Accordion>
<AccordionItem title="Versiuni de pachete">

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Poți folosi opțiunile pentru a personaliza primitivul Sampler. Această secțiune se concentrează pe modul de specificare a opțiunilor primitivelor Qiskit Runtime. Deși interfața metodei `run()` a primitivelor este comună tuturor implementărilor, opțiunile lor nu sunt. Consultă referințele API corespunzătoare pentru informații despre opțiunile [`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) și [`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html).

<span id="pass-options"></span>
## Setarea opțiunilor Sampler
Poți seta opțiunile la inițializarea Sampler, după inițializarea Sampler sau le poți actualiza după ce Sampler a fost inițializat. Pentru instrucțiuni de utilizare a acestor tehnici, consultă tema [Introducere în opțiuni](/guides/runtime-options-overview#options-precedence).

În plus, poți seta valoarea `shots` în metoda `run()`, după cum este descris în secțiunea următoare.
<span id="run-method"></span>
### Metoda Run()
Singurele valori pe care le poți transmite la `run()` sunt cele definite în interfață, adică `shots`. Aceasta suprascrie orice valoare setată pentru `default_shots` pentru execuția curentă.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

Example:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### Cazuri speciale
<span id="shots"></span>
#### Shots

Metoda `SamplerV2.run` acceptă două argumente: o listă de PUB-uri, fiecare putând specifica o valoare shots specifică PUB-ului, și un argument cheie shots. Aceste valori shots fac parte din interfața de execuție a Sampler și sunt independente de opțiunile Sampler Runtime. Acestea au prioritate față de orice valori specificate ca opțiuni, pentru a respecta abstracția Sampler.

Cu toate acestea, dacă `shots` nu este specificat de niciun PUB sau în argumentul cheie run (sau dacă toate sunt `None`), se utilizează valoarea shots din opțiuni, în special `default_shots`.

Pe scurt, aceasta este ordinea de prioritate pentru specificarea shots în Sampler, pentru orice PUB particular:

1. Dacă PUB specifică shots, folosește această valoare.
2. Dacă argumentul cheie `shots` este specificat în `run`, folosește această valoare.
4. Dacă `twirling` este activat (True implicit), se folosește produsul dintre `num_randomizations` și `shots_per_randomization`, specificat ca [opțiuni `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options).
5. Dacă `sampler.options.default_shots` este specificat, folosește această valoare.

Astfel, dacă shots sunt specificate în toate locurile posibile, se folosește cel cu cea mai mare prioritate (shots specificate în PUB).

Exemplu: